# AlphaZero Training — Time Analysis (Connect4)

This notebook profiles the main components of the AlphaZero training pipeline to identify bottlenecks.

**Structure:**
1. Setup — load config, build net and env
2. Micro-benchmarks — individual operations in isolation
3. MCTS phases — selection / expansion / evaluation / backprop
4. Self-play game — end-to-end timing for one game
5. Training step — batch construction + optimizer step
6. Evaluation — `test_vs_mcts` sample
7. Iteration projection — extrapolate to full training iteration
8. Summary chart

In [ ]:
%matplotlib inline

import cProfile
import io
import pstats
import sys
import time
import timeit
from contextlib import contextmanager
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import torch
import yaml

# Add project root to path if needed
ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Import train after matplotlib is already configured — train.py calls
# matplotlib.use("Agg") at module level, but that is a no-op once pyplot
# has already been imported with the inline backend active.
from giotto.agents.algorithms.alphazero.mcts import AlphaZeroMCTS
from giotto.agents.algorithms.alphazero.net import AlphaZeroNet
from giotto.agents.alphazero import AlphaZeroAgent
from giotto.agents.mcts import MCTSAgent
from giotto.envs.connect4 import Connect4Env
from giotto.utils.text_play import play_n_games

print("PyTorch version:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

## 1. Setup

In [ ]:
CONFIG_PATH = ROOT / "giotto/agents/algorithms/alphazero/config_c4.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

print("Config loaded:")
print(f"  Network: {config['network']['residual_blocks']} res-blocks, {config['network']['channels']} channels")
print(f"  MCTS: {config['mcts']['n_sims']} simulations, cpuct={config['mcts']['cpuct']}")
print(f"  Training: {config['training']['games_per_iteration']} games/iter, batch={config['training']['batch_size']}")

env = Connect4Env()

net = AlphaZeroNet(
    input_size=config["network"]["input_size"],
    value_output_size=config["network"]["value_output_size"],
    policy_output_size=config["network"]["policy_output_size"],
    channels=config["network"]["channels"],
    residual_blocks=config["network"]["residual_blocks"],
)
net.eval()

mcts = AlphaZeroMCTS(
    net=net,
    n_simulations=config["mcts"]["n_sims"],
    cpuct=config["mcts"]["cpuct"],
    dirichlet_alpha=config["mcts"]["dirichlet_alpha"],
    dirichlet_eps=config["mcts"]["dirichlet_epsilon"],
)

N_SIMS = config["mcts"]["n_sims"]
print(f"\nSetup complete. N_SIMS={N_SIMS}")

## 2. Micro-benchmarks

Timing individual operations in isolation to get baseline costs.

In [ ]:
REPEATS = 1000
micro = {}

# ---- env.clone() ----
t = timeit.timeit(lambda: env.clone(), number=REPEATS)
micro["env.clone()"] = t / REPEATS

# ---- env.get_valid_actions() ----
t = timeit.timeit(lambda: env.get_valid_actions(), number=REPEATS)
micro["env.get_valid_actions()"] = t / REPEATS

# ---- env.check_win() ----
t = timeit.timeit(lambda: env.check_win(0), number=REPEATS)
micro["env.check_win()"] = t / REPEATS


# ---- env.step() ----
def _step_reset():
    e = env.clone()
    e.step(4)


t = timeit.timeit(_step_reset, number=REPEATS)
micro["env.step()"] = t / REPEATS

# ---- net.predict() ----
state = env.get_state()
t = timeit.timeit(lambda: net.predict(state), number=REPEATS)
micro["net.predict()"] = t / REPEATS

# ---- net.batch_predict() x32 ----
states_batch = [env.get_state() for _ in range(32)]
t = timeit.timeit(lambda: net.batch_predict(states_batch), number=100)
micro["net.batch_predict(x32)"] = (t / 100) / 32  # per-state cost

# ---- AZNode.select_child() ----
mcts.reset()
root = mcts._build_root(env)
# Run a few sims to populate children with visits
for _ in range(50):
    node = root
    while not node.is_terminal() and not node.is_leaf():
        node = node.select_child(config["mcts"]["cpuct"])
    node.expand()
    _, v = net.predict(node.env.get_state())
    node.backpropagate(float(v))

t = timeit.timeit(lambda: root.select_child(config["mcts"]["cpuct"]), number=REPEATS)
micro["node.select_child()"] = t / REPEATS

# ---- AZNode.backpropagate() ----
child = next(iter(root.children.values()))
t = timeit.timeit(lambda: child.backpropagate(0.5), number=REPEATS)
micro["node.backpropagate()"] = t / REPEATS

print("Micro-benchmark results (µs per call):")
print("-" * 45)
for name, seconds in sorted(micro.items(), key=lambda x: -x[1]):
    print(f"  {name:<30}  {seconds * 1e6:8.2f} µs")

## 3. MCTS phases breakdown

Time each phase of a simulation (selection, expansion, evaluation, backprop) for a mid-game state.

In [ ]:
# Create a mid-game state (5 random moves played)
mid_env = Connect4Env()
mid_env.reset()
for action in [4, 4, 3, 5, 2]:
    mid_env.step(action)

phase_times = {"selection": 0.0, "expansion": 0.0, "evaluation": 0.0, "backprop": 0.0}

# Warm-up root
mcts.reset()
mcts_root = mcts._build_root(mid_env)

N_PROFILE_SIMS = N_SIMS

for _ in range(N_PROFILE_SIMS):
    node = mcts_root

    # SELECTION
    t0 = time.perf_counter()
    while not node.is_terminal() and not node.is_leaf():
        node = node.select_child(config["mcts"]["cpuct"])
    phase_times["selection"] += time.perf_counter() - t0

    if not node.is_terminal():
        # EXPANSION
        t0 = time.perf_counter()
        node.expand()
        phase_times["expansion"] += time.perf_counter() - t0

        # EVALUATION
        t0 = time.perf_counter()
        policy, value = net.predict(node.env.get_state())
        valid_actions = node.env.get_valid_actions()
        raw = np.array([policy[a - 1] for a in valid_actions], dtype=np.float32)
        raw /= raw.sum()
        for action, prob in zip(valid_actions, raw):
            node.children[action].prob = float(prob)
        value = float(value)
        phase_times["evaluation"] += time.perf_counter() - t0
    else:
        value = node.terminal_node_eval(node.env, node.to_play)

    # BACKPROPAGATION
    t0 = time.perf_counter()
    node.backpropagate(value)
    phase_times["backprop"] += time.perf_counter() - t0

total_mcts_time = sum(phase_times.values())
print(f"MCTS phases over {N_PROFILE_SIMS} simulations (total {total_mcts_time:.3f}s):")
print("-" * 55)
for phase, t in phase_times.items():
    pct = 100.0 * t / total_mcts_time
    per_sim = t / N_PROFILE_SIMS * 1000
    print(f"  {phase:<14} {t:6.3f}s  ({pct:5.1f}%)  {per_sim:.3f} ms/sim")

## 4. cProfile — single MCTS run

Full `cProfile` on a single `mcts.run()` call (using config n_sims).

In [ ]:
@contextmanager
def profile_block(sort="cumtime", limit=30):
    pr = cProfile.Profile()
    pr.enable()
    try:
        yield
    finally:
        pr.disable()
        s = io.StringIO()
        pstats.Stats(pr, stream=s).strip_dirs().sort_stats(sort).print_stats(limit)
        print(s.getvalue())


fresh_env = Connect4Env()
fresh_env.reset()
mcts.reset()

with profile_block(sort="tottime", limit=25):
    action, root = mcts.run(fresh_env, temperature=0.0)

## 5. Self-play game timing

Time a complete self-play game and record per-move breakdown.

In [ ]:
N_SAMPLE_GAMES = 3  # small sample for speed

game_times = []
move_counts = []
net.eval()

for g in range(N_SAMPLE_GAMES):
    mcts.reset()
    game_env = Connect4Env()
    game_env.reset()
    move_count = 0
    t_game_start = time.perf_counter()

    while not game_env.done:
        schedule = config["mcts"]["temperature_schedule"]
        temperature = config["mcts"]["temperature"] if move_count < schedule else 0.0
        action, _ = mcts.run(game_env, temperature=temperature)
        game_env.step(action)
        mcts.advance_root(action)
        move_count += 1

    game_times.append(time.perf_counter() - t_game_start)
    move_counts.append(move_count)
    print(f"Game {g+1}: {move_count} moves, {game_times[-1]:.2f}s  ({game_times[-1]/move_count*1000:.0f} ms/move)")

avg_game_time = np.mean(game_times)
avg_moves = np.mean(move_counts)
avg_ms_per_move = avg_game_time / avg_moves * 1000
print(f"\nAverage: {avg_game_time:.2f}s/game, {avg_moves:.1f} moves/game, {avg_ms_per_move:.0f} ms/move")

## 6. Training step timing

In [ ]:
import copy
import random

import torch.nn.functional as F

BATCH_SIZE = config["training"]["batch_size"]


# Generate synthetic records with realistic board states (random pieces placed)
def make_fake_records(n):
    records = []
    rng = np.random.default_rng(42)
    for _ in range(n):
        board = np.full((6, 7), -1, dtype=np.int64)
        n_pieces = rng.integers(0, 20)
        for _ in range(n_pieces):
            col = rng.integers(0, 7)
            empty_rows = np.where(board[:, col] == -1)[0]
            if len(empty_rows) > 0:
                board[empty_rows[0], col] = rng.integers(0, 2)
        policy = rng.dirichlet(np.ones(7)).astype(np.float32)
        records.append(
            {
                "state": [board.copy(), int(rng.integers(0, 2))],
                "policy_target": policy,
                "value_target": float(rng.choice([-1.0, 0.0, 1.0])),
            }
        )
    return records


fake_buffer = make_fake_records(2000)

# Train a deep copy so the original net's BatchNorm running stats are not corrupted
net_train = copy.deepcopy(net)
net_train.train()
optimizer = torch.optim.AdamW(net_train.parameters(), lr=config["training"]["learning_rate"])


def train_step(batch_records):
    boards = np.stack([r["state"][0] for r in batch_records])
    players = np.array([r["state"][1] for r in batch_records])
    current = (boards == players[:, None, None]).astype(np.float32)
    opponent = (boards == (1 - players)[:, None, None]).astype(np.float32)
    x = np.stack([current, opponent], axis=1)
    states = torch.from_numpy(x)
    policy_targets = torch.from_numpy(np.stack([r["policy_target"] for r in batch_records]))
    value_targets = torch.tensor([r["value_target"] for r in batch_records], dtype=torch.float32).unsqueeze(1)

    policy_logits, value_preds = net_train(states)
    log_policy = F.log_softmax(policy_logits, dim=1)
    policy_loss = -(policy_targets * log_policy).sum(dim=1).mean()
    value_loss = F.mse_loss(value_preds, value_targets)
    total_loss = policy_loss + value_loss

    optimizer.zero_grad(set_to_none=True)
    total_loss.backward()
    optimizer.step()
    return float(total_loss.item())


# Warm-up
for _ in range(3):
    train_step(random.sample(fake_buffer, BATCH_SIZE))

# Timed run
N_STEPS = 50
step_times = []
for _ in range(N_STEPS):
    batch = random.sample(fake_buffer, BATCH_SIZE)
    t0 = time.perf_counter()
    train_step(batch)
    step_times.append(time.perf_counter() - t0)

avg_step = np.mean(step_times)
print(f"Training step (batch={BATCH_SIZE}):")
print(f"  avg = {avg_step*1000:.1f} ms")
print(f"  p50 = {np.percentile(step_times, 50)*1000:.1f} ms")
print(f"  p95 = {np.percentile(step_times, 95)*1000:.1f} ms")

## 7. Evaluation — `test_vs_mcts` sample

Time a small number of evaluation games to extrapolate to the configured `n_matches`.

In [ ]:
N_EVAL_SAMPLE = 2  # small sample — extrapolate to full
N_MATCHES_FULL = int(config["eval"]["n_matches"])

# Use a fresh net so this cell is independent of training-step side effects
# (training on all-zero boards corrupts BatchNorm running stats, causing NaN probs)
net_eval = AlphaZeroNet(
    input_size=config["network"]["input_size"],
    value_output_size=config["network"]["value_output_size"],
    policy_output_size=config["network"]["policy_output_size"],
    channels=config["network"]["channels"],
    residual_blocks=config["network"]["residual_blocks"],
)
net_eval.eval()

az_agent = AlphaZeroAgent(
    game=config["game"],
    simulations=config["mcts"]["n_sims"],
    cpuct=config["mcts"]["cpuct"],
    net=net_eval,
)
mcts_agent = MCTSAgent(
    simulations=config["mcts"]["n_sims"],
    cpuct=config["mcts"]["cpuct"],
)
agents = [az_agent, mcts_agent]

t0 = time.perf_counter()
play_n_games(N_EVAL_SAMPLE, env.clone(), agents, invert_starts=True)
eval_sample_time = time.perf_counter() - t0

eval_per_game = eval_sample_time / N_EVAL_SAMPLE
eval_full_est = eval_per_game * N_MATCHES_FULL

print(f"Evaluation ({N_EVAL_SAMPLE} games): {eval_sample_time:.2f}s  →  {eval_per_game:.2f}s/game")
print(f"Estimated for {N_MATCHES_FULL} matches (full eval): {eval_full_est:.1f}s  ({eval_full_est/60:.1f} min)")

## 8. Iteration time projection

Extrapolate measured times to a full training iteration.

In [ ]:
GAMES_PER_ITER = int(config["training"]["games_per_iteration"])
BUFFER_SIZE = int(config["training"]["buffer_size"])
training_steps_per_iter = int(config["training"].get("training_steps_per_iteration", 0))

# Estimate steps: if not set, trainer uses len(buffer) // batch_size (capped at buffer_size)
if training_steps_per_iter <= 0:
    # After a few iterations, buffer is near full
    est_steps = min(BUFFER_SIZE, GAMES_PER_ITER * int(avg_moves) * 2) // BATCH_SIZE
else:
    est_steps = training_steps_per_iter

t_selfplay = avg_game_time * GAMES_PER_ITER
t_training = avg_step * est_steps
t_eval = eval_full_est if config["eval"]["play_vs_mcts"] else 0.0
t_total = t_selfplay + t_training + t_eval

projection = {
    "Self-play": t_selfplay,
    "Training steps": t_training,
    "Eval vs MCTS": t_eval,
}

print("Projected time per iteration (single process):")
print("-" * 50)
for label, t in projection.items():
    pct = 100.0 * t / t_total
    print(f"  {label:<20} {t:7.1f}s  ({pct:.1f}%)")
print("-" * 50)
print(f"  {'TOTAL':<20} {t_total:7.1f}s  ({t_total/60:.1f} min)")
print()
print(f"  Self-play: {GAMES_PER_ITER} games × {avg_game_time:.2f}s/game")
print(f"  Training: {est_steps} steps × {avg_step*1000:.1f}ms/step")
print(f"  Eval: {N_MATCHES_FULL} games × {eval_per_game:.2f}s/game")

## 9. Summary charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("AlphaZero Connect4 — Time Analysis", fontsize=13, fontweight="bold")

# --- Chart 1: Iteration breakdown ---
ax = axes[0]
labels = list(projection.keys())
values = [projection[l] for l in labels]
colors = ["#e74c3c", "#3498db", "#2ecc71"]
bars = ax.barh(labels, values, color=colors, edgecolor="white")
ax.set_xlabel("Time (s)")
ax.set_title("Time per iteration\n(single process)")
for bar, v in zip(bars, values):
    ax.text(
        bar.get_width() + max(values) * 0.01, bar.get_y() + bar.get_height() / 2, f"{v:.1f}s", va="center", fontsize=9
    )
ax.set_xlim(0, max(values) * 1.15)

# --- Chart 2: MCTS phases ---
ax = axes[1]
phase_labels = list(phase_times.keys())
phase_vals = [phase_times[k] for k in phase_labels]
phase_colors = ["#9b59b6", "#e67e22", "#e74c3c", "#1abc9c"]
wedges, texts, autotexts = ax.pie(
    phase_vals,
    labels=phase_labels,
    colors=phase_colors,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.75,
)
for at in autotexts:
    at.set_fontsize(9)
ax.set_title(f"MCTS phases\n({N_PROFILE_SIMS} simulations)")

# --- Chart 3: Micro-benchmarks ---
ax = axes[2]
mb_labels = list(micro.keys())
mb_vals_us = [micro[k] * 1e6 for k in mb_labels]
sorted_pairs = sorted(zip(mb_vals_us, mb_labels), reverse=True)
mb_vals_sorted, mb_labels_sorted = zip(*sorted_pairs)
bars = ax.barh(mb_labels_sorted, mb_vals_sorted, color="#34495e", edgecolor="white")
ax.set_xlabel("Time (µs)")
ax.set_title("Micro-benchmarks\n(µs per call)")
ax.set_xscale("log")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}"))
for bar, v in zip(bars, mb_vals_sorted):
    ax.text(v * 1.05, bar.get_y() + bar.get_height() / 2, f"{v:.1f}", va="center", fontsize=8)

plt.tight_layout()
plt.show()
plt.savefig("time_analysis.png")